# DataCo Supply Chain Analysis

**Question:** Where is this business losing money to operational failure, and are those failures concentrated or systemic?

**Dataset:** DataCo Smart Supply Chain, 180,519 order line items, 53 fields.

**Approach:** SQL analysis via DuckDB and JupySQL, with Power BI for visualization.

### Note on grain

The dataset is one row per **order line item**, not one row per order. Order level
counts require `DISTINCT "Order Id"`. Row counts overstate order volume. This governs
every aggregation below.

## Setup

In [ ]:
import pandas as pd

df = pd.read_csv(
    r"C:\Users\ibrah\Downloads\archive (8)\DataCoSupplyChainDataset.csv",
    encoding="latin-1"
)

In [ ]:
import duckdb

conn = duckdb.connect()
%load_ext sql
%sql conn

### Shape and completeness

In [ ]:
print("Rows, columns:", df.shape)
print()
print("Null counts (non-zero only):")
print(df.isna().sum()[df.isna().sum() > 0])
print()
print("Departments:", df['Department Id'].nunique())
print("Categories:", df['Category Id'].nunique())

Order Zipcode (155,679 nulls) and Product Description (180,519 nulls) are unusable
and are excluded from all analysis.

---

# Section 1: Delivery performance

The dataset records both a scheduled shipping window and the actual days taken, which
allows the delivery problem to be decomposed into a fulfillment question and a
scheduling question.

## 1.1 Overall delivery status

Establishes the baseline before breaking the number down.

In [ ]:
%%sql
SELECT "Delivery Status",
       COUNT("Order Id") * 100.0 / SUM(COUNT("Order Id")) OVER() AS "Share of orders %"
FROM df
GROUP BY "Delivery Status"
ORDER BY "Share of orders %" DESC

**Finding:** Roughly 55% of orders are delivered late. This is the headline number
the rest of the section explains.

## 1.2 Late delivery by market

If lateness were a regional logistics problem, rates would vary by market.

In [ ]:
%%sql
SELECT "Market",
       SUM("Order Item Total") AS "Total sales in market",
       COUNT("Order Id") AS "Order count",
       SUM(CASE WHEN "Delivery Status" = 'Late delivery' THEN 1 ELSE 0 END) AS "Late count",
       SUM(CASE WHEN "Delivery Status" = 'Late delivery' THEN 1 ELSE 0 END) * 100.0
           / COUNT("Order Id") AS "Late %"
FROM df
GROUP BY "Market"
ORDER BY "Late %" DESC

**Finding:** The late rate is nearly flat across markets. LATAM has the lowest rate
on the highest volume. Africa and USCA run higher on much smaller volumes, but the
spread between best and worst is narrow.

**Implication:** This is not a regional problem. Regional intervention would not move
the number, so the cause sits upstream of geography.

## 1.3 Late delivery by department

Same test applied to product division rather than geography.

In [ ]:
%%sql
SELECT "Department Id",
       SUM(CASE WHEN "Delivery Status" = 'Advance shipping' THEN 1 ELSE 0 END) * 100.0
           / COUNT(*) AS "Advance %",
       SUM(CASE WHEN "Delivery Status" = 'Shipping on time' THEN 1 ELSE 0 END) * 100.0
           / COUNT(*) AS "On time %",
       SUM(CASE WHEN "Delivery Status" = 'Late delivery' THEN 1 ELSE 0 END) * 100.0
           / COUNT(*) AS "Late %",
       SUM(CASE WHEN "Delivery Status" = 'Shipping canceled' THEN 1 ELSE 0 END) * 100.0
           / COUNT(*) AS "Cancelled %"
FROM df
GROUP BY "Department Id"
ORDER BY "Late %" DESC

**Finding:** Late rates are similarly flat across departments. Neither geography nor
product division explains the failure.

## 1.4 Scheduled versus actual shipping days by mode

This is the query that identifies the mechanism. Comparing the promise against the
delivery isolates whether the failure is in fulfillment speed or in the commitment.

In [ ]:
%%sql
SELECT "Shipping Mode",
       AVG("Days for shipment (scheduled)") AS "Scheduled days",
       AVG("Days for shipping (real)") AS "Actual days",
       AVG("Days for shipping (real)") - AVG("Days for shipment (scheduled)") AS "Gap"
FROM df
GROUP BY "Shipping Mode"
ORDER BY "Gap" DESC

**Finding:** Scheduled days are fixed integers. Every order within a tier receives an
identical promise regardless of distance, market, or product. Actual transit time
converges near 4 days for both Second Class and Standard Class.

| Shipping Mode | Scheduled | Actual | Gap |
|---|---|---|---|
| Second Class | 2.0 | 4.0 | +2.0 |
| First Class | 1.0 | 2.0 | +1.0 |
| Same Day | 0.0 | 0.5 | +0.5 |
| Standard Class | 4.0 | 4.0 | -0.004 |

Standard Class is the only mode whose promise matches operational reality. Every tier
sold as faster than Standard misses its window, and Second Class misses widest at 2 full
days despite being priced below First Class.

**Implication:** The tiers reflect differentiated promises against a network that ships
at one speed, not differentiated capability. Correcting the scheduled windows on Second
and First Class would resolve most late deliveries with no operational change. It would
also expose that expedited shipping delivers little real speed advantage, which is a
commercial question rather than a logistics one.

## 1.5 Failed deliveries by shipping mode

Confirms the pattern using late and cancelled orders combined.

In [ ]:
%%sql
SELECT "Shipping Mode",
       SUM(CASE WHEN "Delivery Status" IN ('Late delivery', 'Shipping canceled')
           THEN 1 ELSE 0 END) AS "Late or cancelled",
       SUM(CASE WHEN "Delivery Status" IN ('Late delivery', 'Shipping canceled')
           THEN 1 ELSE 0 END) * 100.0 / COUNT("Order Id") AS "Failure %"
FROM df
GROUP BY "Shipping Mode"
ORDER BY "Failure %" DESC

---

# Section 2: Profitability

Two questions: what does the business earn, and what does operational failure cost it.

## 2.1 Foregone profit from cancelled orders

Framed as foregone rather than lost. Cancelled orders never generated revenue, so this
is profit that did not materialize rather than a write-off. Revenue is deliberately not
used here, since unearned revenue is not a loss.

In [ ]:
%%sql
SELECT
    SUM(CASE WHEN "Delivery Status" = 'Shipping canceled'
        THEN "Benefit per order" ELSE 0 END) AS "Foregone profit",
    SUM("Benefit per order") AS "Total profit",
    SUM(CASE WHEN "Delivery Status" = 'Shipping canceled'
        THEN "Benefit per order" ELSE 0 END) * 100.0
        / SUM("Benefit per order") AS "Foregone as % of total"
FROM df

**Finding:** Cancellations forgo $160,482 against $3,966,903 in total profit, or
about 4.05%.

**Implication:** This is the only failure mode in the dataset carrying a concrete dollar
figure, which makes it the most defensible target for intervention.

## 2.2 Profit margin by department

Margin is computed as total profit divided by total revenue. Averaging the per order
profit ratio was rejected because that average weights every order equally regardless of
size, biasing the result toward small orders.

In [ ]:
%%sql
SELECT "Department Id",
       SUM("Order Item Total") AS "Revenue",
       SUM("Order Profit Per Order") AS "Profit",
       SUM("Order Profit Per Order") * 100.0
           / SUM("Order Item Total") AS "Margin %"
FROM df
GROUP BY "Department Id"
ORDER BY "Margin %" DESC

**Finding:** Margins range from about 8% to 13%. The low end is occupied by
departments with negligible revenue, so the volume weighted picture is tighter than the
raw range suggests. No department is structurally unprofitable and none is a standout.

## 2.3 Profit margin by category

In [ ]:
%%sql
SELECT "Category Name",
       SUM("Order Item Total") AS "Revenue",
       SUM("Order Profit Per Order") AS "Profit",
       SUM("Order Item Discount") AS "Discount given",
       SUM("Order Profit Per Order") * 100.0
           / SUM("Order Item Total") AS "Margin %"
FROM df
GROUP BY "Category Name"
ORDER BY "Margin %" DESC

---

# Section 3: Commercial policy

Whether the business differentiates its pricing and discounting by customer value.

## 3.1 Share of orders receiving a discount

In [ ]:
%%sql
SELECT "Department Id",
       COUNT(CASE WHEN "Order Item Discount Rate" != 0 THEN 1 END) * 100.0
           / COUNT("Order Id") AS "Discounted orders %"
FROM df
GROUP BY "Department Id"
ORDER BY "Discounted orders %" DESC

**Finding:** Roughly 94% of orders across every department carry a discount.

## 3.2 Discount received versus total spend

If discounting were a lever, larger customers would receive better terms. Plotting these
two columns against each other produces a flat line.

In [ ]:
%%sql
SELECT "Customer Id",
       SUM("Order Item Total") AS "Total spent",
       SUM("Order Item Discount") AS "Total discount received",
       SUM("Order Item Discount") * 100.0
           / SUM("Order Item Total") AS "Effective discount rate %"
FROM df
GROUP BY "Customer Id"
ORDER BY "Total spent" DESC
LIMIT 50

**Finding:** Effective discount rates sit near 10.2% regardless of customer size. The
largest buyers receive no better terms than the smallest.

**Implication:** Discounting functions as a blanket price reduction rather than a lever.
There is no volume incentive, no loyalty tier, and no negotiated pricing.

## 3.3 Customer segment mix

In [ ]:
df['Customer Segment'].value_counts(normalize=True)

---

# Section 4: Revenue structure

Where revenue concentrates, and what that implies about risk.

## 4.1 Customer concentration

In [ ]:
%%sql
SELECT "Customer Id", "Customer Segment",
       SUM("Order Item Total") AS "Total spent",
       COUNT("Order Id") AS "Order count",
       SUM("Order Item Total") * 100.0
           / SUM(SUM("Order Item Total")) OVER() AS "% of revenue"
FROM df
GROUP BY "Customer Id", "Customer Segment"
ORDER BY "% of revenue" DESC
LIMIT 30

**Finding:** The top 30 customers each contribute around 2% of revenue, with no single
account materially larger than another.

**Implication:** No key account risk. Losing any one customer would not be material.

## 4.2 Departmental concentration

Revenue and category breadth per department, to distinguish genuine concentration from
small departments that happen to carry one product line.

In [ ]:
%%sql
SELECT "Department Id",
       COUNT(DISTINCT "Category Id") AS "Category count",
       SUM("Order Item Total") AS "Revenue",
       SUM("Order Item Total") * 100.0
           / SUM(SUM("Order Item Total")) OVER() AS "% of total revenue"
FROM df
GROUP BY "Department Id"
ORDER BY "Revenue" DESC

**Finding:** Four departments account for roughly 92% of revenue. The remaining seven
are marginal, three of them under $100K.

The single category departments are the smallest by revenue, so their apparent
concentration is a size artifact rather than a strategic choice. This is worth stating
explicitly, since reading concentration percentages without checking the underlying
volume would produce a misleading conclusion.

## 4.3 Category concentration within departments

In [ ]:
%%sql
WITH dept_cat AS (
    SELECT "Department Id", "Category Id",
           SUM("Order Item Total") AS "Category sales",
           SUM(SUM("Order Item Total")) OVER (PARTITION BY "Department Id")
               AS "Department sales"
    FROM df
    GROUP BY "Department Id", "Category Id"
)
SELECT "Department Id", "Category Id", "Category sales", "Department sales",
       "Category sales" * 100.0 / "Department sales" AS "Category % of department"
FROM dept_cat
WHERE "Category sales" * 100.0 / "Department sales" > 50
ORDER BY "Category % of department" DESC

## 4.4 Order geography

Whether orders are domestic or cross border relative to the fulfilling department.

In [ ]:
%%sql
SELECT "Order Country",
       SUM("Order Item Total") AS "Revenue",
       COUNT("Order Id") AS "Order count",
       COUNT("Order Id") * 100.0 / SUM(COUNT("Order Id")) OVER()
           AS "% of all orders"
FROM df
GROUP BY "Order Country"
ORDER BY "Revenue" DESC
LIMIT 20

---

# Questions the data could not answer

Two lines of inquiry were opened and closed because the dataset does not support them.
Recording them prevents the same questions being asked twice, and prevents conclusions
being drawn from columns that carry no variation.

## Inventory availability

Product Status was intended to identify stockouts and slow moving inventory.

In [ ]:
df['Product Status'].value_counts()

**Result:** Product Status is 0 for all 180,519 rows. No product is ever recorded as
unavailable, so no stockout, restocking, or inventory analysis is possible.

## Bulk versus retail pricing

Order Item Quantity was intended to test whether bulk purchases receive better terms.

In [ ]:
print("Max order quantity:", df['Order Item Quantity'].max())
print()
print(df['Order Item Quantity'].value_counts().sort_index())

**Result:** Maximum order quantity is 5. There are no bulk orders in this dataset, so
bulk versus retail comparisons are not meaningful.

## Possible extensions

Modeling late delivery as a classification problem would test whether lateness is
predictable at order time or effectively random given the fixed scheduling policy. Time
series decomposition would establish whether the 55% rate is stable or trending. Both
build on the scheduling finding rather than restate it.